# B2-022-probabilistic-latent-models — Practice p22 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** kl-divergence, variational-autoencoder

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

### (a) Diagnosis: posterior collapse

- **KL column:** total KL falls from $0.141$ to $0.000$, and **both** per-latent KLs go to $0.000$ — every posterior $q(z\mid x)$ has become the prior $N(0,I)$, so the code carries no information about $x$.
- **std of `mu` column:** the spread of posterior means across the batch shrinks from $0.31$ to $0.01$ — the encoder maps every input to essentially the same point.
- **recon column vs. the baseline line:** recon settles at $0.599$, essentially the "predict the training mean" baseline $0.598$; the decoder has learned to output the training mean regardless of $z$.

The total fell from $1.012$ to $0.599$ only because the KL was driven to zero and the decoder learned the mean, which a model with no latent at all achieves; a falling total is not evidence of learning when it plateaus at the trivial baseline.

### (b) The reduction defect and hidden $\beta$

The ELBO (unit-variance decoder, constant dropped) requires $\text{recon}=\tfrac12\sum_{j=1}^{8}(x_j-\hat x_j)^2$, a **sum over features**, but the run uses `.mean(dim=1)`, which divides it by $d=8$.
So the run's loss is
$$\tfrac18\,\text{recon}_\text{sum}+\text{KL}=\tfrac18\big(\text{recon}_\text{sum}+8\,\text{KL}\big),$$
i.e. a correct ELBO with a hidden $\beta=d=8$ on the KL (the overall $\tfrac18$ only rescales the gradient). In proper units the plateau reconstruction is $0.599\times8=4.792$ and the baseline $0.598\times8=4.784$: with an 8-fold over-weighted KL, encoding information is not worth its KL price, so the optimizer collapses the posterior.

### (c) Remedy

**Choose (ii):** fix the reduction to the sum over features and keep $\beta=1$, with a linear KL warm-up over the first 100 updates. This restores the correct ELBO weighting (removing the hidden $\beta=8$), and the warm-up lets the encoder start carrying information before the full KL penalty applies — exactly the early phase (updates 0–100) where the log shows the collapse happening.

- **(i) rejected:** keeping the mean-over-features code and setting $\beta=8$ makes the effective weight $8\times8=64$ relative to the correct ELBO; it worsens the collapse it is meant to fix.
- **(iii) rejected:** fixing the reduction but setting $\beta=0.01$ almost removes the KL, so the latent will be used but the posteriors drift far from $N(0,I)$; the objective is no longer a bound on $\log p(x)$, prior samples land in regions the decoder never saw (bad generation), and the model behaves like a noisy plain autoencoder that can overfit.

### (d) Post-fix checks, and choosing $\beta$ without held-out rows

After the fix, check on the training (and validation) log:

1. **per-latent KL:** each latent's mean KL $\ge0.1$ nats (neither latent collapsed to 0);
2. **std of `mu` across the batch:** $\ge0.3$, so different inputs get different codes;
3. **reconstruction (sum-over-features units):** well below the train-mean baseline, e.g. $\le0.6\times(0.598\times8)\approx2.87$.

$\beta$ and the warm-up length are hyperparameters; if they are tuned to make the held-out numbers look good, the held-out rows influence a training decision (selection leakage) and the final held-out score is optimistically biased and no longer an independent estimate. They must be chosen on the training rows or a validation subset carved from them, and the held-out rows evaluated once at the end.

In [ ]:
d, k = 8, 2
hidden_beta = d                                  # mean over features = (1/d) * sum
baseline_mean_units = 0.598
plateau_recon_mean_units = 0.599
baseline_sum_units = baseline_mean_units * d
plateau_sum_units = plateau_recon_mean_units * d
effective_beta_option_i = 8 * hidden_beta
log_kl = [0.141, 0.028, 0.006, 0.002, 0.001, 0.000]
log_mu_std = [0.31, 0.12, 0.05, 0.02, 0.01, 0.01]
log_total = [1.012, 0.668, 0.618, 0.603, 0.600, 0.599]
CHOICE = "ii"
print(hidden_beta, baseline_sum_units, plateau_sum_units, effective_beta_option_i)

### Answer check

In [ ]:
assert hidden_beta == 8
assert abs(baseline_sum_units - 4.784) < 1e-9 and abs(plateau_sum_units - 4.792) < 1e-9
assert effective_beta_option_i == 64
assert log_kl[-1] == 0.0 and log_mu_std[-1] <= 0.01
assert abs(log_total[-1] - baseline_mean_units) <= 0.002      # total ends at the trivial baseline
assert CHOICE == "ii"